# Speech-to-Text with the Azure Speech SDK

**Speech recognition** (speech-to-text / STT) converts spoken audio into text. This notebook does *single-shot* recognition — one utterance at a time with `recognize_once()` — starting from a WAV file and then extending to:

- Robustly handling every result outcome (recognized / no-match / canceled)
- Detailed output with **confidence scores** and word-level timestamps
- Recognizing live speech from the **microphone**
- Automatically **detecting the spoken language**

> For long or streaming audio, see the notebook on continuous recognition.

In [ ]:
import os
from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk

In [ ]:
load_dotenv()

In [ ]:
key = os.getenv("AZURE_SPEECH_KEY")
region = os.getenv("AZURE_SPEECH_REGION")

## Recognizing a single utterance

`AudioConfig(filename="Data/FightMilk.wav")` points the recognizer at a WAV file (16 kHz mono PCM works best). The `SpeechRecognizer` binds credentials to that audio source. `recognize_once()` blocks until one utterance is transcribed and returns a `SpeechRecognitionResult`; `result.text` is the transcript.

In [ ]:
speech_config = speechsdk.SpeechConfig(subscription=key, region=region)
audio_config = speechsdk.audio.AudioConfig(filename="Data/FightMilk.wav")

In [ ]:
speech_client = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

In [ ]:
result = speech_client.recognize_once()

In [ ]:
result.text

> **Why the transcript stops early.** `recognize_once()` is **single-shot** recognition: it captures exactly *one* utterance and returns as soon as it detects the first significant pause in speech (and it caps at roughly 15 seconds of audio regardless). So on a longer file like `FightMilk.wav`, you only get the text up to that first pause — everything after it is never transcribed. This is by design, not a bug: single-shot is meant for short, command-style inputs (a spoken query, a form field, one dialogue turn).
>
> To transcribe an entire file or a live stream end to end, you use **continuous recognition** instead — `start_continuous_recognition()` with a `recognized` event handler that fires once per utterance until you stop it. That pattern is covered in the continuous-recognition notebook referenced at the top.

## Inspecting the result object

Beyond `text`, the result exposes `reason` (a `ResultReason` enum value describing the outcome) plus details like `cancellation_details` and `no_match_details`. The next cells explore this: `dir(result)` lists everything available, `result.reason` shows this call's outcome, and `list(speechsdk.ResultReason)` enumerates every possible outcome across recognition, synthesis, and translation.

In [ ]:
dir(result)

In [ ]:
result.reason

In [ ]:
list(speechsdk.ResultReason)

## Robustly handling the result

The single line `result.text` silently returns an empty string when nothing was recognized or when the call failed. Production code should branch on `result.reason` so it can distinguish success from a no-match (silence/noise) and from a cancellation (bad key, region, or audio problem).

In [ ]:
# recognize_once returns one result. Always branch on result.reason so the code
# behaves sensibly whether speech was found, not found, or the call failed.
result = speech_client.recognize_once()

if result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print(f"Recognized: {result.text}")
elif result.reason == speechsdk.ResultReason.NoMatch:
    # Audio was processed but no speech matched (silence, noise, wrong language).
    print(f"No speech recognized: {result.no_match_details}")
elif result.reason == speechsdk.ResultReason.Canceled:
    # The request was stopped — usually a bad key/region or an audio problem.
    details = result.cancellation_details
    print(f"Canceled: {details.reason}")
    if details.reason == speechsdk.CancellationReason.Error:
        print(f"Error details: {details.error_details}")

## Detailed output — confidence scores and word timings

By default the service returns just the display text. Set `output_format = Detailed` and call `request_word_level_timestamps()` to get back ranked alternatives (`NBest`), a **confidence** score per hypothesis, and per-word start offsets. The full response is available as JSON on `result.json`.

In [ ]:
import json

# A fresh config that asks the service for DETAILED results (confidence + alternatives)
# plus word-level timestamps. Both must be set BEFORE the recognizer is created.
detailed_config = speechsdk.SpeechConfig(subscription=key, region=region)
detailed_config.output_format = speechsdk.OutputFormat.Detailed
detailed_config.request_word_level_timestamps()

detailed_audio = speechsdk.audio.AudioConfig(filename="Data/FightMilk.wav")
detailed_client = speechsdk.SpeechRecognizer(
    speech_config=detailed_config, audio_config=detailed_audio
)
result = detailed_client.recognize_once()

# result.json holds the full service response. Parse it for the ranked hypotheses.
payload = json.loads(result.json)
best = payload["NBest"][0]  # NBest is ordered best-first
print(f"Display text : {best['Display']}")
print(f"Confidence   : {best['Confidence']:.3f}\n")

print("First few word timings (word @ start offset in seconds):")
for word in best["Words"][:5]:
    # Offset/Duration are in 100-ns ticks; divide by 10,000,000 for seconds.
    print(f"  {word['Word']:12} @ {word['Offset'] / 1e7:.2f}s")

## Recognizing from the microphone

Swap the file `AudioConfig` for `use_default_microphone=True` to transcribe live speech. `recognize_once()` captures a single utterance (up to ~15 seconds) and stops at the first pause.

> **Note:** this cell needs a working microphone and will block waiting for you to speak. Skip it if you're running headless.

In [ ]:
# use_default_microphone=True captures live audio instead of reading a file.
mic_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
mic_client = speechsdk.SpeechRecognizer(
    speech_config=speechsdk.SpeechConfig(subscription=key, region=region),
    audio_config=mic_config,
)

print("Speak now...")
result = mic_client.recognize_once()  # blocks until you finish speaking
print(f"You said: {result.text}")

## Automatic language detection

When you don't know the spoken language up front, provide a list of candidate languages and let the service pick. `recognize_once` supports up to four candidates; the chosen language is read back via `AutoDetectSourceLanguageResult`. This is useful for multilingual call centers and document-intake pipelines.

In [ ]:
# Offer a candidate list; the service detects which one is actually spoken.
auto_detect = speechsdk.languageconfig.AutoDetectSourceLanguageConfig(
    languages=["en-US", "es-ES", "de-DE", "fr-FR"]  # up to 4 for recognize_once
)
detect_client = speechsdk.SpeechRecognizer(
    speech_config=speechsdk.SpeechConfig(subscription=key, region=region),
    auto_detect_source_language_config=auto_detect,
    audio_config=speechsdk.audio.AudioConfig(filename="Data/FightMilk.wav"),
)
result = detect_client.recognize_once()

# Wrap the result to read back which language was detected.
detected = speechsdk.AutoDetectSourceLanguageResult(result).language
print(f"Detected language: {detected}")
print(f"Text: {result.text}")